In [ ]:
# ===== Block 1: Imports =====
import os
import random
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import sklearn

from torch.utils.data import Dataset

from torch_geometric.loader import DataLoader

In [ ]:
# ===== Block 2: Settings =====
CACHE_PATH = "../data/graphs_cached.pt"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("====Versions:====")
print("torch:\t", torch.__version__)
print("PyG:\t", torch_geometric.__version__)
print("scikit-learn", sklearn.__version__)
print("device:\t", DEVICE)
if torch.cuda.is_available():
    print("CUDA:\t", torch.version.cuda)

In [ ]:
class DonorAcceptorDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df: pd.DataFrame = df.reset_index(drop=True)
        #TODO: define programatically
        self.num_features = 13
        self.num_graph_features = 250
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        donor = row["D_graph"]
        acceptor = row["A_graph"]
        
        return donor, acceptor

In [ ]:
if os.path.exists(CACHE_PATH):
    df = torch.load(CACHE_PATH, weights_only=False)
else:
    raise(NotImplementedError)

In [ ]:
# mols = pd.concat([df['D_mol'], df['A_mol']] )

In [ ]:
n = len(df)
idx = np.arange(n)
np.random.shuffle(idx)

cut = int(0.9 * n)
train_df = df.iloc[idx[:cut]].reset_index(drop=True)
val_df   = df.iloc[idx[cut:]].reset_index(drop=True)

train_ds = DonorAcceptorDataset(train_df)
val_ds   = DonorAcceptorDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print("Train/Val:", len(train_ds), len(val_ds))

In [ ]:
train_ds[0][0]

# VGAE
[Bresson and Laurent, 2019](https://ui.adsabs.harvard.edu/link_gateway/2019arXiv190603412B/doi:10.48550/arXiv.1906.03412)

## TODO:
- [ ] Redo node attributes to assign atomic number to a dictionary
- [ ] Redo edge attributes to include None type


In [ ]:
# ==== Encoder ===== 
from torch_geometric.nn import GATConv

# May redo this to use ConvNet as per the paper
class MolecularEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, graph_in_dim, latent_dim, hidden_dim):
        super().__init__()

        # Broadcast graph features
        in_dim = node_in_dim + graph_in_dim

        self.conv1 = GATConv(in_dim, hidden_dim, edge_dim=edge_in_dim)

        self.conv_mu = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)
        self.conv_logstd = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)

    def forward(self, x, edge_index, edge_attr, graph_attr, batch):
        
        # Broadcast graph attributes
        graph_attr_batch = graph_attr[batch]
        x = torch.cat([x, graph_attr_batch], dim=-1)

        x = F.relu(self.conv1(x, edge_index, edge_attr))

        mu = self.conv_mu(x, edge_index, edge_attr)
        logstd = self.conv_logstd(x, edge_index, edge_attr)

        return mu, logstd

In [ ]:
# ==== Decoder =====
from torch_geometric.nn.conv.gatv2_conv import GATv2Conv


from torch import Tensor
from torch_geometric.nn import MLP, GATv2Conv


class MolecularDecoder(nn.Module):
    def __init__(self, latent_dim: int, atom_dict, edge_dict, max_mol_size: int, emb_dim: int, gat_hidden_dim=64):
        super().__init__()
        self.atom_dict = atom_dict
        self.edge_dict = edge_dict
        self.max_size: int = max_mol_size
        self.emb_dim: int = emb_dim

        self.boa_mlp: MLP = MLP(in_channels=latent_dim,
                        out_channels=self.atom_dict_size * self.max_size,
                        hidden_channels=64)

        self.node_emb: nn.Embedding = nn.Embedding(num_embeddings=self.atom_dict_size,
                                                    embedding_dim=self.emb_dim)
        
        self.edge_emb: nn.Linear = nn.Linear(in_features=latent_dim,
                                            out_features=latent_dim,
                                            bias=False)

        # May need to change in_channels
        # TODO: change to take variable number of layers
        self.conv1: GATv2Conv = GATv2Conv(in_channels=emb_dim,
                                        out_channels=gat_hidden_dim,
                                        edge_dim=latent_dim)
        self.conv2: GATv2Conv = GATv2Conv(in_channels=gat_hidden_dim,
                                        out_channels=emb_dim,
                                        edge_dim=latent_dim)

        self.edge_mlp: MLP = MLP(in_channels=emb_dim,
                                out_channels=self.edge_dict_size,
                                hidden_channels=64)

        


    def forward(self, z: Tensor):
        # Atom Generation
        z_soft_boa: Tensor = self.boa_mlp(z).reshape([self.atom_dict_size, self.max_size])
        z_boa = z_soft_boa.max(dim=1).indices

        # Bond Generation
        ## Create a fully connected graph
        num_atoms = int(torch.sum(z_boa))
        edge_index = torch.ones([num_atoms, num_atoms], dtype=torch.bool)

        # Node embedding
        # Create vector of repeating node_types according to bag of atoms
        nodes = torch.repeat_interleave(z_boa, z_boa, dim=0)
        x = self.node_emb(nodes)

        ## Edge embedding
        emb_edges = self.edge_emb(z)
        edge_attr = emb_edges.repeat(edge_index.size)

        gat_feats = self.conv1(x, edge_index, edge_attr)
        gat_feats = F.relu(gat_feats)
        gat_feats = self.conv2(gat_feats, edge_index, edge_attr)

        edge_scores = self.edge_mlp(gat_feats)

        # Breaking the Symmetry
        # TODO
        # may need to change this to a self.field to keep consistent across the dataset
        max_duplicate_atoms = int(torch.max(z_boa))

        

    @property
    def atom_dict_size(self) -> int:
        return len(self.atom_dict)
    
    @property
    def edge_dict_size(self) -> int:
        return len(self.edge_dict)
    
    @property
    def max_mol_size(self) -> int:
        return self.max_size

    @property
    def max_edges(self) -> int:
        s = self.max_size
        return s*s